In [16]:
import os
from dotenv import load_dotenv
from groq import Groq
import feedparser
import json

In [10]:
load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))


In [ ]:
try:
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        max_tokens=100,
        messages=[
            {"role": "user", "content": "Say hello in one short sentence."}
        ]
    )
    print(response.choices[0].message.content)
except Exception as e:
    print(str(e))

Hello!


In [14]:
feed_url = "https://economictimes.indiatimes.com/markets/rssfeeds/1977021501.cms"
feed = feedparser.parse(feed_url)

for entry in feed.entries[:5]:
    print(entry.title)

Nifty mega caps are going nowhere. Why Samco CIO Umesh Mehta put 75% of his Flexicap fund beyond largecaps
MSCI rebalancing on Monday: Adani Energy, Groww set for big inflows; RIL, Jio Financial may face outflows
Quality Stock Investing: 8 mistakes that can ruin your portfolio and how to avoid them
Warren Buffett turns 96: Top 10 investing lessons from the Oracle of Omaha
SBI, SBI Capital Markets plan 1% NSE stake sale via IPO: Bank Chairman C S Setty


In [15]:
keywords = ["CDSL", "Reliance", "RIL", "TCS", "HDFC Bank", "HDFC"]

relevant_headlines = [
    entry.title for entry in feed.entries
    if any(keyword.lower() in entry.title.lower() for keyword in keywords)
]

print(relevant_headlines)

['MSCI rebalancing on Monday: Adani Energy, Groww set for big inflows; RIL, Jio Financial may face outflows', '7 of top 10 firms shed Rs 1.13 lakh cr in m-cap; Airtel, Reliance worst hit', 'Reliance Jio IPO: Google, Meta among 6 investors to retain stakes in Rs 37,700 crore offer', 'Reliance Jio IPO: 7 risk factors investors should know as firm gets Sebi nod for Rs 37,000-crore issue']


In [17]:
headlines_text = "\n".join(relevant_headlines)

prompt = f"""Score the sentiment of each of the following headlines on a scale from -1 (very negative) to +1 (very positive) for the stock they mention.

Headlines:
{headlines_text}

Return ONLY a JSON array, no other text, in this exact format:
[{{"headline": "...", "sentiment": 0.0}}, ...]
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    max_tokens=500,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

[
  {"headline": "MSCI rebalancing on Monday: Adani Energy, Groww set for big inflows; RIL, Jio Financial may face outflows", "sentiment": -0.3},
  {"headline": "7 of top 10 firms shed Rs 1.13 lakh cr in m-cap; Airtel, Reliance worst hit", "sentiment": -0.8},
  {"headline": "Reliance Jio IPO: Google, Meta among 6 investors to retain stakes in Rs 37,700 crore offer", "sentiment": 0.6},
  {"headline": "Reliance Jio IPO: 7 risk factors investors should know as firm gets Sebi nod for Rs 37,000-crore issue", "sentiment": -0.5}
]


In [18]:
sentiment_data = json.loads(response.choices[0].message.content)
print(sentiment_data)

for item in sentiment_data:
    print(f"{item['sentiment']:+.2f}  {item['headline']}")

[{'headline': 'MSCI rebalancing on Monday: Adani Energy, Groww set for big inflows; RIL, Jio Financial may face outflows', 'sentiment': -0.3}, {'headline': '7 of top 10 firms shed Rs 1.13 lakh cr in m-cap; Airtel, Reliance worst hit', 'sentiment': -0.8}, {'headline': 'Reliance Jio IPO: Google, Meta among 6 investors to retain stakes in Rs 37,700 crore offer', 'sentiment': 0.6}, {'headline': 'Reliance Jio IPO: 7 risk factors investors should know as firm gets Sebi nod for Rs 37,000-crore issue', 'sentiment': -0.5}]
-0.30  MSCI rebalancing on Monday: Adani Energy, Groww set for big inflows; RIL, Jio Financial may face outflows
-0.80  7 of top 10 firms shed Rs 1.13 lakh cr in m-cap; Airtel, Reliance worst hit
+0.60  Reliance Jio IPO: Google, Meta among 6 investors to retain stakes in Rs 37,700 crore offer
-0.50  Reliance Jio IPO: 7 risk factors investors should know as firm gets Sebi nod for Rs 37,000-crore issue


In [21]:
def get_news_sentiment(tickers, keywords, feed_url="https://economictimes.indiatimes.com/markets/rssfeeds/1977021501.cms"):
    feed = feedparser.parse(feed_url)

    relevant_headlines = [
        entry.title for entry in feed.entries
        if any(keyword.lower() in entry.title.lower() for keyword in keywords)
    ]

    if not relevant_headlines:
        return None

    headlines_text = "\n".join(relevant_headlines)

    prompt = f"""Score the sentiment of each of the following headlines on a scale from -1 (very negative) to +1 (very positive) for the stock they mention.

Headlines:
{headlines_text}

Return ONLY a JSON array, no other text, in this exact format:
[{{"headline": "...", "sentiment": 0.0}}, ...]
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        max_tokens=800,
        messages=[{"role": "user", "content": prompt}]
    )

    raw_content = response.choices[0].message.content
    print("RAW OUTPUT:", raw_content)  # temporary debug line

    # strip markdown code fences if the model added them
    cleaned = raw_content.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        cleaned = cleaned.replace("json", "", 1).strip()

    sentiment_data = json.loads(cleaned)
    avg_sentiment = sum(item["sentiment"] for item in sentiment_data) / len(sentiment_data)

    return {
        "headlines": sentiment_data,
        "average_sentiment": avg_sentiment
    }

In [22]:
result = get_news_sentiment(
    tickers=["CDSL.NS", "RELIANCE.NS", "TCS.NS", "HDFCBANK.NS"],
    keywords=["CDSL", "Reliance", "RIL", "TCS", "HDFC Bank", "HDFC"]
)
print(result)

RAW OUTPUT: [{"headline": "MSCI rebalancing on Monday: Adani Energy, Groww set for big inflows; RIL, Jio Financial may face outflows", "sentiment": 0.0}, {"headline": "7 of top 10 firms shed Rs 1.13 lakh cr in m-cap; Airtel, Reliance worst hit", "sentiment": -0.8}, {"headline": "Reliance Jio IPO: Google, Meta among 6 investors to retain stakes in Rs 37,700 crore offer", "sentiment": 0.7}, {"headline": "Reliance Jio IPO: 7 risk factors investors should know as firm gets Sebi nod for Rs 37,000-crore issue", "sentiment": -0.5}]
{'headlines': [{'headline': 'MSCI rebalancing on Monday: Adani Energy, Groww set for big inflows; RIL, Jio Financial may face outflows', 'sentiment': 0.0}, {'headline': '7 of top 10 firms shed Rs 1.13 lakh cr in m-cap; Airtel, Reliance worst hit', 'sentiment': -0.8}, {'headline': 'Reliance Jio IPO: Google, Meta among 6 investors to retain stakes in Rs 37,700 crore offer', 'sentiment': 0.7}, {'headline': 'Reliance Jio IPO: 7 risk factors investors should know as fir